In [1]:
#Import libraries and load data
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from category_encoders.target_encoder import TargetEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

In [2]:
transactions = pd.read_csv('transactions_fe.csv')
transactions.head()

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,user_avg_amt_24h,is_first_merchant_visit,is_first_city_visit,is_rare_merchant,user_city,user_region,is_same_city,is_same_region,city_changed,city_change_fast
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,179.280,1,1,1,bloemfontein,free state,0,0,0,0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,89.640,1,1,1,bloemfontein,free state,0,0,1,0
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,1967.760,1,1,1,bloemfontein,free state,0,0,0,0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,1722.600,1,0,1,bloemfontein,free state,0,0,1,0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,1433.916,1,1,1,bloemfontein,free state,0,0,1,0


In [3]:
transactions.columns

Index(['id', 'date', 'user_id', 'card_id', 'use_chip', 'merchant_id',
       'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_fraud',
       'description', 'id_y', 'client_id_y', 'card_brand', 'card_type',
       'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued',
       'credit_limit', 'acct_open_date', 'year_pin_last_changed',
       'card_on_dark_web', 'id.1', 'current_age', 'retirement_age',
       'birth_year', 'birth_month', 'gender', 'address', 'per_capita_income',
       'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards',
       'is_negative_amount', 'abs_amount', 'transaction_hour',
       'transaction_dayofweek', 'is_weekend', 'is_night', 'is_rush_hour',
       'time_since_last_txn', 'user_avg_amount', 'user_std_amount',
       'amount_to_avg_ratio', 'txn_count_24h', 'txn_count_7d', 'card_age_days',
       'is_high_value', 'user_hour_mean', 'user_hour_std', 'unusual_hour',
       'hour_median', 'robust_z_hour', 'unusual_hour_flag', 'i

In [4]:
df = transactions.copy()

In [5]:
# already exists but make explicit
df["new_city_flag"] = (df["user_city"] != df["merchant_city"]).astype(int)

# interaction-based affinity
df["new_city_and_high_value"] = (
    df["new_city_flag"] & df["is_high_value"]
).astype(int)


In [6]:
# how often user has seen this merchant (proxy)
df["user_merchant_familiarity"] = (
    1 - df["is_first_merchant_visit"]
)

In [7]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,is_rare_merchant,user_city,user_region,is_same_city,is_same_region,city_changed,city_change_fast,new_city_flag,new_city_and_high_value,user_merchant_familiarity
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,1,bloemfontein,free state,0,0,0,0,1,0,0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,1,bloemfontein,free state,0,0,1,0,1,1,0
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,1,bloemfontein,free state,0,0,0,0,1,0,0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,1,bloemfontein,free state,0,0,1,0,1,0,0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,1,bloemfontein,free state,0,0,1,0,1,0,0
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,1,bloemfontein,free state,0,0,1,1,1,0,0
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,1,bloemfontein,free state,0,0,1,0,1,0,1
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,1,bloemfontein,free state,0,0,1,0,1,1,1
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,1,bloemfontein,free state,0,0,1,0,1,0,0
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,1,bloemfontein,free state,0,0,0,0,1,1,0


In [10]:
df['user_merchant_familiarity'].value_counts()

user_merchant_familiarity
1    8312177
0     303356
Name: count, dtype: int64

In [13]:
df["location_mismatch"] = (
    (~df["is_same_city"].astype(bool)) &
    (~df["is_same_region"].astype(bool))
).astype(int)

In [15]:
df["txn_velocity_24h"] = (
    df["user_txn_count_24h"] /
    (df["user_txn_count_7d"] / 7 + 1e-6)
)

In [17]:
df["amt_velocity_24h"] = (
    df["user_amt_sum_24h"] /
    (df["user_avg_amount"] + 1e-6)
)

In [18]:
df["rapid_repeat_txn"] = (df["time_since_last_txn"] < 300).astype(int)  # 5 min

In [20]:
df["debt_to_income"] = (
    df["total_debt"] / (df["yearly_income"] + 1e-6)
)

In [21]:
df['debt_to_income'].value_counts()

debt_to_income
0.000000    454972
1.585616     31426
1.349326     28240
0.010924     27532
1.026198     26901
             ...  
1.954813      1361
1.437206      1223
1.571795      1214
1.854814      1131
2.369143       396
Name: count, Length: 1155, dtype: int64

In [22]:
df.columns

Index(['id', 'date', 'user_id', 'card_id', 'use_chip', 'merchant_id',
       'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_fraud',
       'description', 'id_y', 'client_id_y', 'card_brand', 'card_type',
       'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued',
       'credit_limit', 'acct_open_date', 'year_pin_last_changed',
       'card_on_dark_web', 'id.1', 'current_age', 'retirement_age',
       'birth_year', 'birth_month', 'gender', 'address', 'per_capita_income',
       'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards',
       'is_negative_amount', 'abs_amount', 'transaction_hour',
       'transaction_dayofweek', 'is_weekend', 'is_night', 'is_rush_hour',
       'time_since_last_txn', 'user_avg_amount', 'user_std_amount',
       'amount_to_avg_ratio', 'txn_count_24h', 'txn_count_7d', 'card_age_days',
       'is_high_value', 'user_hour_mean', 'user_hour_std', 'unusual_hour',
       'hour_median', 'robust_z_hour', 'unusual_hour_flag', 'i

In [25]:
df["user_amt_z"] = (
    (df["abs_amount"] - df["user_avg_amount"]) /
    (df["user_std_amount"] + 1e-6)
)

In [26]:
df["extreme_amt_z"] = (df["user_amt_z"].abs() > 3).astype(int)

In [27]:
df['extreme_amt_z'].value_counts()

extreme_amt_z
0    8478742
1     136791
Name: count, dtype: int64

In [28]:
df["combined_anomaly_score"] = (
    df["user_amt_z"].abs() +
    df["robust_z_hour"].abs() +
    df["amount_to_avg_ratio"]
)

In [29]:
df['combined_anomaly_score'].value_counts()

combined_anomaly_score
2.236566e+00    257
2.124151e+00    190
2.348982e+00    166
2.011736e+00    161
1.899321e+00     99
               ... 
2.193330e+00      1
6.498000e+07      1
2.700000e+08      1
2.080800e+08      1
1.278000e+07      1
Name: count, Length: 8614323, dtype: int64

In [30]:
df["hour_deviation"] = (
    df["transaction_hour"] -
    df["hour_median"]
).abs()

In [31]:
df["off_hours_high_value"] = (
    df["is_night"] & df["is_high_value"]
).astype(int)

In [33]:
df['errors'].unique()

array(['No Errors', 'bad pin', 'technical glitch', 'insufficient balance',
       'bad card number', 'bad zipcode', 'bad expiration', 'bad cvv',
       'bad pin,insufficient balance',
       'insufficient balance,technical glitch',
       'bad cvv,technical glitch', 'bad pin,technical glitch',
       'bad cvv,insufficient balance', 'bad card number,bad expiration',
       'bad expiration,insufficient balance',
       'bad zipcode,technical glitch',
       'bad card number,insufficient balance',
       'bad expiration,technical glitch', 'bad card number,bad cvv',
       'bad expiration,bad cvv', 'bad card number,technical glitch',
       'bad zipcode,insufficient balance',
       'bad card number,bad expiration,insufficient balance'],
      dtype=object)

In [34]:
HIGH_RISK_CATEGORIES = [
    "Digital Goods",
    "Betting",
    "Money Transfer",
    "Precious Stones",
    "Electronics Stores",
    "Telecommunication Services",
    "Computer Network Services",
]
df["high_risk_category"] = (
    df["description"]
    .str.contains("|".join(HIGH_RISK_CATEGORIES), case=False, na=False)
).astype(int)

In [36]:
df = df.sort_values("date")

df["is_first_category_visit"] = (
    df.groupby("user_id")["description"]
      .transform(lambda x: ~x.duplicated())
      .astype(int)
)

In [37]:
df["user_category_avg_amt"] = (
    df.groupby(["user_id", "description"])["abs_amount"]
      .transform("mean")
)

df["category_amt_ratio"] = (
    df["abs_amount"] /
    (df["user_category_avg_amt"] + 1e-6)
)


In [38]:
df["has_error"] = (df["errors"] != "No Errors").astype(int)

In [39]:
df["error_count"] = df["errors"].str.count(",").fillna(0) + 1
df.loc[df["errors"] == "No Errors", "error_count"] = 0

In [40]:
SENSITIVE_ERRORS = [
    "bad pin", "bad cvv", "bad expiration", "bad card number"
]

df["verification_failure"] = (
    df["errors"]
    .str.contains("|".join(SENSITIVE_ERRORS), na=False)
).astype(int)

In [41]:
df["balance_issue"] = (
    df["errors"].str.contains("insufficient balance", na=False)
).astype(int)

In [ ]:
'''df = df.sort_values("date")
g = df.groupby("user_id")

df["errors_24h"] = (
    g["has_error"]
    .rolling("24h")
    .sum()
    .shift(1)
)'''

In [43]:
df["risky_category_fast"] = (
    df["high_risk_category"] &
    (df["txn_velocity_24h"] > 2)
).astype(int)

In [44]:
df["new_category_high_value"] = (
    df["is_first_category_visit"] &
    df["is_high_value"]
).astype(int)


In [45]:
df["cred_guessing_pattern"] = (
    df["verification_failure"] &
    (df["time_since_last_txn"] < 300)
).astype(int)


In [46]:
df["is_online_txn"] = (df["use_chip"] == 'online transaction').astype(int)

In [48]:
df["verification_failure"].value_counts()

verification_failure
0    8581380
1      34153
Name: count, dtype: int64

In [49]:
df["online_verification_failure"] = (
    df["verification_failure"] &
    df["is_online_txn"]
).astype(int)

In [50]:
df["online_verification_failure"].value_counts()

online_verification_failure
0    8602391
1      13142
Name: count, dtype: int64

In [51]:
df["online_cred_guess_high_value"] = (
    df["online_verification_failure"] &
    df["is_high_value"]
).astype(int)

In [52]:
df["online_cred_guess_high_value"].value_counts()

online_cred_guess_high_value
0    8614473
1       1060
Name: count, dtype: int64

In [ ]:
'''df = df.sort_values("date")
g = df.groupby("user_id")

df["online_verif_fail_24h"] = (
    g["online_verification_failure"]
    .rolling("24h")
    .sum()
    .shift(1)
)'''

In [55]:
df.columns.tolist()

['id',
 'date',
 'user_id',
 'card_id',
 'use_chip',
 'merchant_id',
 'merchant_city',
 'merchant_state',
 'zip',
 'mcc',
 'errors',
 'is_fraud',
 'description',
 'id_y',
 'client_id_y',
 'card_brand',
 'card_type',
 'card_number',
 'expires',
 'cvv',
 'has_chip',
 'num_cards_issued',
 'credit_limit',
 'acct_open_date',
 'year_pin_last_changed',
 'card_on_dark_web',
 'id.1',
 'current_age',
 'retirement_age',
 'birth_year',
 'birth_month',
 'gender',
 'address',
 'per_capita_income',
 'yearly_income',
 'total_debt',
 'credit_score',
 'num_credit_cards',
 'is_negative_amount',
 'abs_amount',
 'transaction_hour',
 'transaction_dayofweek',
 'is_weekend',
 'is_night',
 'is_rush_hour',
 'time_since_last_txn',
 'user_avg_amount',
 'user_std_amount',
 'amount_to_avg_ratio',
 'txn_count_24h',
 'txn_count_7d',
 'card_age_days',
 'is_high_value',
 'user_hour_mean',
 'user_hour_std',
 'unusual_hour',
 'hour_median',
 'robust_z_hour',
 'unusual_hour_flag',
 'is_iqr_outlier',
 'is_z_outlier',
 'inc

In [56]:
df.to_csv('transactions_fe_v2.csv', index=False)